# Lab 5 — Data that moves

**Accompanies Lecture 6 (optimal transport, delay embeddings, curvature).**

You build the tools from the lecture, check them on the examples we used, and then find the places where they break.

| # | Exercise | What you build | When |
|---|---|---|---|
| 1 | Distances that move | `tv_hist`, `mmd_gauss`, `w2_1d` | session, 15 min |
| 2 | Matching, Hungarian and Sinkhorn | `brute_force_cost`, `sorted_cost`, `closest_first_cost`, `w2sq_assignment`, `sinkhorn_cost`, `sinkhorn_cost_log` | session, 20 min |
| 3 | The middle, the average, and the blind spot | `interpolate_transport`, `interpolate_mixture`, `detour`, `barycentre_1d` | session, 20 min |
| 4 | Delay embedding from scratch | `mutual_information`, `first_minimum`, `fnn_fraction`, `twonn_curve`, `twonn_consecutive_and_random` | session, 30 min |
| 5 | Windows as clouds | `aaft`, `window_cloud`, `window_distances` | take-home |
| 6 | Curvature on graphs | `ollivier_curvature` | take-home |

Every exercise ends with an `assert` cell. If your code is correct, the notebook runs from top to bottom without an error. Cells marked **GIVEN** are complete; cells with `# TODO` are yours. The seeds are fixed; change `SEED` below to another number to check that your solutions do not depend on luck.

You need the packages `pot` (the optimal transport library) and `networkx`; the next cell installs them if they are missing.

In [ ]:
try:
    import ot
except ImportError:
    %pip install -q pot
    import ot
try:
    import networkx as nx
except ImportError:
    %pip install -q networkx
    import networkx as nx
import warnings, itertools, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.integrate import solve_ivp
from scipy.spatial.distance import cdist
from scipy.special import logsumexp
from scipy.stats import rankdata
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
NAVY, TEAL, AMBER, MUTE = "#1E2761", "#1C7293", "#E8871E", "#6E7691"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 0
print("numpy", np.__version__, "| POT", ot.__version__, "| networkx", nx.__version__)

## Exercise 1 — Distances that move

In the lecture four candidates tried to say how far apart two clouds are. You build three of them and put them to the test.

You write:

1. `tv_hist(x, y, bins)`: the total variation between two samples. Use one common range (the minimum and the maximum of both samples together), `np.histogram` with `bins` bins, turn both counts into probabilities, and return half the sum of the absolute differences.
2. `mmd_gauss(x, y, width)`: the maximum mean discrepancy with the Gaussian kernel $k(a,b)=\exp\!\big(-(a-b)^2/2w^2\big)$, that is $\sqrt{\overline{k(x,x')}+\overline{k(y,y')}-2\,\overline{k(x,y)}}$, where each bar is the mean over **all** pairs (a point with itself included).
3. `w2_1d(x, y)`: the Wasserstein-2 distance between two samples of the same size on a line: sort both and compare in order.

In [ ]:
def tv_hist(x, y, bins):
    """Total variation between two 1D samples, from histograms with `bins` bins on a common range."""
    # TODO
    raise NotImplementedError

def mmd_gauss(x, y, width):
    """Maximum mean discrepancy with a Gaussian kernel of the given width (mean over all pairs)."""
    # TODO
    raise NotImplementedError

def w2_1d(x, y):
    """Wasserstein-2 distance between two 1D samples of the same size."""
    # TODO
    raise NotImplementedError

**Part A.** Two blobs (standard deviation 0.5) at distance $d$, 2000 points each. The histogram cell has width 0.1, the kernel width is 1.

In [ ]:
# GIVEN: the table of the lecture
rng = np.random.default_rng(SEED); n = 2000
ds = [0.25, 0.5, 1, 2, 4, 8, 16]; rows = []
for d in ds:
    x, y = rng.normal(0, 0.5, n), rng.normal(d, 0.5, n)
    bins = int(np.ceil((max(x.max(), y.max()) - min(x.min(), y.min())) / 0.1))
    rows.append((d, tv_hist(x, y, bins), mmd_gauss(x, y, 1.0), w2_1d(x, y)))
print(f"{'d':>6s} {'TV':>7s} {'MMD':>7s} {'W2':>8s}")
for r in rows: print(f"{r[0]:6.2f} {r[1]:7.3f} {r[2]:7.3f} {r[3]:8.3f}")

In [ ]:
# CHECK: Part A
tv, mm, ww = [np.array([r[k] for r in rows]) for k in (1, 2, 3)]
assert (tv <= 1 + 1e-12).all() and (tv >= 0).all(), "total variation lies between 0 and 1 (half the sum of the absolute differences)"
assert (tv[4:] >= 0.99).all(), "total variation saturates at 1 once the clouds stop overlapping"
assert abs(ww[-1] - 16) < 0.3 and abs(ww[2] - 1) < 0.1, "W2 between two shifted copies of a cloud is the shift"
assert (np.diff(ww) > 0).all(), "W2 keeps growing with the displacement"
assert 1.1 < mm[-1] < 1.45 and abs(mm[-1] - mm[-2]) < 0.05, "MMD saturates beyond the kernel width"
print("Exercise 1, Part A passed.")

**Part B.** The same distribution twice: two independent samples of $N(0,1)$, 1000 points each. The true distance is zero. Which bin width is the right one?

In [ ]:
# GIVEN: total variation of two samples of the SAME distribution, as the number of bins grows (mean of 20 repetitions)
rng = np.random.default_rng(SEED + 1)
bins_list = [5, 10, 30, 100, 300, 1000, 5000]
tv_same = [np.mean([tv_hist(rng.normal(size=1000), rng.normal(size=1000), b) for _ in range(20)]) for b in bins_list]
print("  ".join(f"{b}: {v:.3f}" for b, v in zip(bins_list, tv_same)))
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.bar(range(len(bins_list)), tv_same, color=NAVY); ax.set_xticks(range(len(bins_list))); ax.set_xticklabels(bins_list)
ax.set_xlabel("number of histogram bins"); ax.set_ylabel("total variation"); plt.tight_layout(); plt.show()

In [ ]:
# CHECK: Part B
assert (np.diff(tv_same) > 0).all(), "the number moves in one direction as the bins get finer"
assert tv_same[0] < 0.08 and tv_same[-1] > 0.6, "from almost nothing to almost 1, for the same distribution"
print("Exercise 1, Part B passed.")

**Part C.** A ring (radius 1) and a Gaussian blob (standard deviation $1/\sqrt2$) have the same mean and the same covariance. The function `w2_2d` is given: it computes the exact $W_2$ with the POT library (you build the same thing from scratch in Exercise 2).

In [ ]:
# GIVEN: the ring and the blob
def w2_2d(A, B):
    n, m = len(A), len(B)
    return float(np.sqrt(ot.emd2(np.ones(n) / n, np.ones(m) / m, ot.dist(A, B))))

def ring_and_blob(n, rng):
    th = rng.uniform(0, 2 * np.pi, n)
    return np.column_stack([np.cos(th), np.sin(th)]), rng.normal(0, np.sqrt(0.5), (n, 2))

rng = np.random.default_rng(SEED + 2); n = 1000
R1, B1 = ring_and_blob(n, rng); R2, B2 = ring_and_blob(n, rng)
w_rb, f_ring, f_blob = w2_2d(R1, B1), w2_2d(R1, R2), w2_2d(B1, B2)
print(f"means: {R1.mean(0).round(3)} and {B1.mean(0).round(3)}; covariance diagonals: {np.cov(R1.T).diagonal().round(3)} and {np.cov(B1.T).diagonal().round(3)}")
print(f"W2(ring, blob) = {w_rb:.3f};  sampling floor: two rings {f_ring:.3f}, two blobs {f_blob:.3f}")

In [ ]:
# CHECK: Part C
assert np.abs(R1.mean(0) - B1.mean(0)).max() < 0.15 and np.linalg.norm(np.cov(R1.T) - np.cov(B1.T)) < 0.2, "same mean, same covariance"
assert w_rb > 2.5 * max(f_ring, f_blob), "W2 sees the difference of shape that the moments cannot see"
print("Exercise 1 passed.")

*Discuss:* in Part A, which of the three distances would still tell you how far day 5 is from day 3 when the clouds do not overlap? In Part B, what would you change in your analysis if a collaborator reported "total variation 0.4" without the bin width?

## Exercise 2 — Matching, Hungarian and Sinkhorn

With equal weights, optimal transport is a **matching problem**: pair every point of one cloud with a point of the other so that the total cost is smallest.

You write:

1. `brute_force_cost(a, b)`: for two 1D samples of the same (small) size, the smallest total squared distance over **all** pairings (`itertools.permutations`).
2. `sorted_cost(a, b)`: the total squared distance when both samples are sorted and paired in order.
3. `closest_first_cost(a, b)`: the greedy rule of the lecture: find the closest unmatched pair, match it, remove both, repeat. Return the total squared distance.
4. `w2sq_assignment(A, B)`: for clouds in any dimension, the **mean** squared distance of the optimal pairing, i.e. $W_2^2$. Build the matrix `sqdist(A, B)` (given) and use `linear_sum_assignment`.
5. `sinkhorn_cost(C, eps, n_iter=2000)`: Sinkhorn's algorithm on the cost matrix `C` with uniform weights: $K=e^{-C/\varepsilon}$, then alternate $u=a/(Kv)$, $v=b/(K^\top u)$. Return the transport cost $\sum_{ij}P_{ij}C_{ij}$ of the plan $P=\mathrm{diag}(u)\,K\,\mathrm{diag}(v)$.
6. `sinkhorn_cost_log(C, eps, n_iter=1500)`: the same in the **log domain**, with the potentials $f,g$: $f=-\varepsilon\,\mathrm{logsumexp}_j\big((g_j-C_{ij})/\varepsilon+\log b_j\big)$, $g=-\varepsilon\,\mathrm{logsumexp}_i\big((f_i-C_{ij})/\varepsilon+\log a_i\big)$, and $P_{ij}=\exp\!\big((f_i+g_j-C_{ij})/\varepsilon\big)\,a_i b_j$.

In [ ]:
def sqdist(A, B):                                   # GIVEN
    return cdist(A, B, "sqeuclidean")

def brute_force_cost(a, b):
    """Smallest total squared distance over all pairings of two 1D samples of the same size."""
    # TODO
    raise NotImplementedError

def sorted_cost(a, b):
    """Total squared distance when both samples are sorted and paired in order."""
    # TODO
    raise NotImplementedError

def closest_first_cost(a, b):
    """Greedy: match the closest unmatched pair, remove both, repeat. Total squared distance."""
    # TODO
    raise NotImplementedError

def w2sq_assignment(A, B):
    """W2 squared between two clouds of the same size: the mean squared distance of the optimal pairing."""
    # TODO
    raise NotImplementedError

def sinkhorn_cost(C, eps, n_iter=2000):
    """Transport cost <P, C> of Sinkhorn's plan with uniform weights (multiplicative form)."""
    # TODO
    raise NotImplementedError

def sinkhorn_cost_log(C, eps, n_iter=1500):
    """The same in the log domain: no exponential of a large negative number is ever formed."""
    # TODO
    raise NotImplementedError

**Part A.** One dimension: random instances of 7 points (for the brute force) and of 8 points (for the greedy rule).

In [ ]:
# GIVEN: sorting against brute force, and the greedy rule
rng = np.random.default_rng(SEED + 10)
inst7 = [(rng.normal(0, 1, 7), rng.normal(0.5, 1, 7)) for _ in range(30)]
bf = np.array([brute_force_cost(a, b) for a, b in inst7]); so = np.array([sorted_cost(a, b) for a, b in inst7])
inst8 = [(rng.normal(0, 1, 8), rng.normal(0.5, 1, 8)) for _ in range(300)]
cf = np.array([closest_first_cost(a, b) for a, b in inst8]); so8 = np.array([sorted_cost(a, b) for a, b in inst8])
rnd = np.array([np.sum((a - rng.permutation(b)) ** 2) for a, b in inst8])
frac_worse = float((cf > so8 + 1e-9).mean())
print(f"sorted equals the brute-force optimum in {int(np.isclose(bf, so).sum())} of 30 instances")
print(f"mean cost over 300 instances of 8 points: random {rnd.mean():.2f}, closest first {cf.mean():.2f}, sorted {so8.mean():.2f};  closest first is strictly worse than sorted in {100 * frac_worse:.0f}% of them")

In [ ]:
# CHECK: Part A
assert np.allclose(bf, so), "on a line the optimal pairing is the sorted one"
assert (cf >= so8 - 1e-9).all(), "no pairing can beat the optimum"
assert frac_worse > 0.25, "the greedy rule is often strictly suboptimal"
assert rnd.mean() > 1.5 * so8.mean(), "a random pairing is much worse than the optimum"
print("Exercise 2, Part A passed.")

**Part B.** Any dimension: your assignment-based $W_2^2$ against the one of POT, and against sorting on a line.

In [ ]:
# GIVEN: the Hungarian algorithm against POT
rng = np.random.default_rng(SEED + 11)
A, B = rng.normal(size=(200, 2)), rng.normal(size=(200, 2)) + [1.0, 1.0]
mine, pot_ = w2sq_assignment(A, B), float(ot.emd2(np.ones(200) / 200, np.ones(200) / 200, ot.dist(A, B)))
a1, b1 = rng.normal(size=300), rng.normal(size=300) * 2 + 1
line_assign, line_sort = w2sq_assignment(a1[:, None], b1[:, None]), float(np.mean((np.sort(a1) - np.sort(b1)) ** 2))
print(f"W2^2: {mine:.10f} (yours)  {pot_:.10f} (POT);   on a line: assignment {line_assign:.8f}, sorting {line_sort:.8f}")

In [ ]:
# CHECK: Part B
assert abs(mine - pot_) < 1e-9, "the optimal assignment is the optimal transport plan for equal weights"
assert abs(line_assign - line_sort) < 1e-9, "on a line the optimal assignment is sorting"
print("Exercise 2, Part B passed.")

**Part C.** Sinkhorn against the exact value. Two clouds of 300 points in the plane, the second shifted by 1 in both coordinates. The error you will measure is the price of the entropy penalty.

In [ ]:
# GIVEN: Sinkhorn's error against the exact W2^2
rng = np.random.default_rng(SEED + 12)
A, B = rng.normal(size=(300, 2)), rng.normal(size=(300, 2)) + [1.0, 1.0]; C = sqdist(A, B)
exact = w2sq_assignment(A, B); eps_list = [0.5, 0.1, 0.02]
sk = {e: sinkhorn_cost(C, e) for e in eps_list}; err = {e: (sk[e] - exact) / exact for e in eps_list}
print(f"exact W2^2 = {exact:.4f}")
for e in eps_list: print(f"  Sinkhorn eps = {e:5.2f}: {sk[e]:.4f}  ({100 * err[e]:+.1f}%)")

In [ ]:
# CHECK: Part C
assert 0.10 < err[0.5] < 0.35, "a strong penalty costs a lot of accuracy"
assert 0.01 < err[0.1] < 0.08
assert abs(err[0.02]) < 0.02
assert err[0.5] > err[0.1] > abs(err[0.02]), "the error shrinks with eps"
print("Exercise 2, Part C passed.")

**Part D. The trap.** A small $\varepsilon$ gives an accurate plan, so why not use $\varepsilon=0.01$? Run the multiplicative version and look at what comes out, then use the log-domain version.

In [ ]:
# GIVEN: the underflow trap (150 points to keep it quick)
rng = np.random.default_rng(SEED + 13)
A, B = rng.normal(size=(150, 2)), rng.normal(size=(150, 2)) + [1.0, 1.0]; C150 = sqdist(A, B); exact150 = w2sq_assignment(A, B)
with np.errstate(all="ignore"):
    naive = sinkhorn_cost(C150, 0.01); zero_frac = float((np.exp(-C150 / 0.01) == 0).mean())
logv = sinkhorn_cost_log(C150, 0.01)
print(f"eps = 0.01: {100 * zero_frac:.0f}% of the entries of exp(-C/eps) underflow to 0;  multiplicative result: {naive};  log-domain result: {logv:.4f}  (exact {exact150:.4f}, {100 * (logv - exact150) / exact150:+.1f}%)")

In [ ]:
# CHECK: Part D
assert not np.isfinite(naive), "the multiplicative version divides 0 by 0"
assert np.isfinite(logv) and abs(logv - exact150) / exact150 < 0.03, "the log-domain version works at the same eps"
print("Exercise 2 passed.")

In [ ]:
# GIVEN: the timing question of the lecture (no assert): 1000 points
rng = np.random.default_rng(SEED + 14)
A, B = rng.normal(size=(1000, 2)), rng.normal(size=(1000, 2)) + [1.0, 1.0]; C1k = sqdist(A, B)
t0 = time.time(); ex = w2sq_assignment(A, B); t_assign = time.time() - t0
t0 = time.time(); ex_pot = float(ot.emd2(np.ones(1000) / 1000, np.ones(1000) / 1000, C1k)); t_pot = time.time() - t0
print(f"1000 points, exact W2^2 = {ex:.4f}: scipy assignment {t_assign:.2f} s, POT network simplex {t_pot:.2f} s")
for eps, its in ((0.5, 100), (0.1, 500), (0.02, 3000)):
    t0 = time.time(); s1 = sinkhorn_cost(C1k, eps, n_iter=its); t_sk = time.time() - t0
    print(f"   Sinkhorn eps = {eps:4.2f}, {its:4d} iterations: {t_sk:5.2f} s, error {100 * (s1 - ex) / ex:+5.1f}%")

*Discuss:* in the timing table, at which $\varepsilon$ is Sinkhorn clearly faster than the exact solver, and what does that cost in accuracy? Which exact solver (SciPy's assignment or POT's network simplex) would you use for 2000 points? Why does the multiplicative version fail at a small $\varepsilon$, and what exactly is the problem (which number is zero)?

## Exercise 3 — The middle, the average, and the blind spot

You have two snapshots, and you want the cloud in between, and the average cloud.

You write:

1. `interpolate_transport(A, B, perm, t)`: the points $(1-t)A_i+tB_{\mathrm{perm}[i]}$, where `perm` is a pairing.
2. `interpolate_mixture(A, B, t, rng)`: the "average of the densities": `round((1 - t) * n)` points drawn at random from `A`, completed to `n` points with points drawn at random from `B`.
3. `detour(A, B, perm, t)`: the length of the path through the middle cloud $M_t$ (built with your `interpolate_transport`) divided by the straight distance: $\big(W_2(A,M_t)+W_2(M_t,B)\big)/W_2(A,B)$. Use the given `w2`.
4. `barycentre_1d(samples, weights=None)`: the Wasserstein barycentre of 1D samples of the same size: the weighted average of the sorted samples.

In [ ]:
def optimal_perm(A, B):                             # GIVEN
    r, c = linear_sum_assignment(cdist(A, B, "sqeuclidean")); perm = np.empty(len(A), int); perm[r] = c
    return perm

def w2(A, B):                                       # GIVEN: exact W2 between clouds of the same size
    C = cdist(A, B, "sqeuclidean"); r, c = linear_sum_assignment(C)
    return float(np.sqrt(C[r, c].mean()))

def interpolate_transport(A, B, perm, t):
    """Point i of A moves the fraction t of the way towards B[perm[i]]."""
    # TODO
    raise NotImplementedError

def interpolate_mixture(A, B, t, rng):
    """The average of the densities: round((1 - t) n) random points of A and n - round((1 - t) n) random points of B."""
    # TODO
    raise NotImplementedError

def detour(A, B, perm, t):
    """(W2(A, M_t) + W2(M_t, B)) / W2(A, B) for the middle cloud M_t of the pairing perm."""
    # TODO
    raise NotImplementedError

def barycentre_1d(samples, weights=None):
    """W2 barycentre of 1D samples of equal size: the weighted average of the sorted samples."""
    # TODO
    raise NotImplementedError

**Part A.** A blob and the same blob moved by 4, and a third cloud that we measured in between (the truth). Which candidate for the middle is closer to it?

In [ ]:
# GIVEN: the held-out snapshot at t = 1/2
rng = np.random.default_rng(SEED + 20); n = 500
A = rng.normal(0, 0.5, (n, 2)); B = rng.normal(0, 0.5, (n, 2)) + [4.0, 0.0]; truth = rng.normal(0, 0.5, (n, 2)) + [2.0, 0.0]
perm = optimal_perm(A, B); Mt = interpolate_transport(A, B, perm, 0.5); Mx = interpolate_mixture(A, B, 0.5, rng)
d_ot, d_mix, d_ends = w2(truth, Mt), w2(truth, Mx), w2(A, B)
print(f"W2 to the true cloud at t = 1/2: points moved halfway {d_ot:.3f}, average of the densities {d_mix:.3f}   (the end clouds are {d_ends:.3f} apart)")
fig, ax = plt.subplots(1, 3, figsize=(10, 2.6), sharex=True, sharey=True)
for a, P, t, c in zip(ax, (truth, Mx, Mt), ("truth", "average of the densities", "points moved halfway"), (AMBER, MUTE, TEAL)):
    a.scatter(*A.T, s=2, color="#CADCFC"); a.scatter(*B.T, s=2, color="#CADCFC"); a.scatter(*P.T, s=5, color=c); a.set_title(t, fontsize=9); a.set_aspect("equal")
plt.tight_layout(); plt.show()

In [ ]:
# CHECK: Part A
assert d_ot < 0.25, "the transport interpolation is at the sampling floor"
assert d_mix > 1.0, "the average of the densities keeps the two blobs where they were"
assert d_mix > 5 * d_ot
assert interpolate_transport(A, B, perm, 0.0).shape == A.shape and np.allclose(interpolate_transport(A, B, perm, 0.0), A) and np.allclose(interpolate_transport(A, B, perm, 1.0), B[perm])
print("Exercise 3, Part A passed.")

**Part B. Is the path a geodesic?** If the middle cloud lies on a shortest path, the detour factor is exactly 1. Try the optimal pairing, a random pairing, and, for a ring that turns by 90°, the pairing "point $i$ goes to its own image".

In [ ]:
# GIVEN: the detour factors
rng = np.random.default_rng(SEED + 21); n = 100
A = rng.normal(0, 1.0, (n, 2)); B = rng.normal(0, 1.0, (n, 2)) + [4.0, 0.0]
det_opt, det_rand = detour(A, B, optimal_perm(A, B), 0.5), detour(A, B, rng.permutation(n), 0.5)
ang = rng.uniform(0, 2 * np.pi, n); R0 = 3 * np.column_stack([np.cos(ang), np.sin(ang)]); R1 = 3 * np.column_stack([np.cos(ang + np.pi / 2), np.sin(ang + np.pi / 2)])
det_same = detour(R0, R1, np.arange(n), 0.5); w_ring = w2(R0, R1); cost_same = float(np.mean(np.sum((R0 - R1) ** 2, axis=1)))
print(f"translating blob: detour with the optimal pairing {det_opt:.6f}, with a random pairing {det_rand:.3f}")
print(f"ring turned by 90 degrees: W2 between the two clouds {w_ring:.3f};  'same index' pairing: mean squared distance {cost_same:.1f}, detour {det_same:.3f}")

In [ ]:
# CHECK: Part B
assert abs(det_opt - 1) < 1e-6, "with the optimal pairing the middle lies on a shortest path"
assert det_rand > 1.01, "with any other pairing the path through the middle is longer than the straight distance"
assert w_ring < 1.2 and cost_same > 15, "the two rings are almost the same cloud, but each point moves a long way"
assert det_same > 1.5, "following each point is not a geodesic of the space of clouds"
print("Exercise 3, Part B passed.")

**Part C.** The average of three distributions: $N(0,1)$, $N(4,0.5)$, $N(8,2)$, 4000 samples each.

In [ ]:
# GIVEN: barycentre against mixture
rng = np.random.default_rng(SEED + 22)
S = np.array([rng.normal(m, s, 4000) for m, s in ((0, 1), (4, 0.5), (8, 2))])
bary = barycentre_1d(S); mix_sd = S.reshape(-1).std()
print(f"barycentre: mean {bary.mean():.3f}, sd {bary.std():.3f} (the average of the three sds is 1.167);  mixture: sd {mix_sd:.3f}")

In [ ]:
# CHECK: Part C
assert abs(bary.mean() - 4) < 0.1 and abs(bary.std() - 7 / 6) < 0.1, "the barycentre has the average position and the average width"
assert mix_sd > 3.3, "the mixture is three humps"
w = np.array([0.5, 0.25, 0.25]); b2 = barycentre_1d(S, w)
assert abs(b2.mean() - (0.5 * 0 + 0.25 * 4 + 0.25 * 8)) < 0.1, "weights move the barycentre"
print("Exercise 3 passed.")

*Discuss:* in Part B, a ring that turns has $W_2\approx 0$ to its former self. What does this say about what can be learned from snapshots of unlabeled points, and what would you need to see the rotation?

## Exercise 4 — Delay embedding from scratch

A system with three hidden variables, observed through one: the first coordinate $x(t)$ of the Lorenz system, sampled every 0.02 time units. The functions `lorenz_x`, `delay` and `two_nn` (from Lab 5) are given.

You write:

1. `mutual_information(x, lag, bins=32)`: the mutual information between $x_t$ and $x_{t+\mathrm{lag}}$ from a two-dimensional histogram (`np.histogram2d` with `bins` bins on the range of $x$), $\sum p_{ab}\ln\frac{p_{ab}}{p_a p_b}$ over the non-empty cells; and `first_minimum(mi)`: the lag of the first minimum, where `mi[k]` is the value at lag `k + 1` (the first lag after which the values increase).
2. `fnn_fraction(x, m, tau, ratio=10.0, n=1500, theiler=200, seed=0)`: the fraction of false nearest neighbours. Draw `n` random starting times (sorted), form the $m$-dimensional delay vectors, find for each the nearest neighbour **whose time differs by more than `theiler`**, and call the pair false if the next coordinate differs by more than `ratio` times their distance: $|x_{i+m\tau}-x_{j+m\tau}|/d_{ij}>\mathrm{ratio}$.
3. `twonn_curve(x, tau, ms, n=1500, seed=0)`: for every $m$ in `ms`, the TwoNN dimension of `n` random delay vectors (mean of three draws).
4. `twonn_consecutive_and_random(x, m, tau, n_consecutive=4000, n_random=1500, seed=0)`: the TwoNN dimension of the first `n_consecutive` delay vectors, and of `n_random` random ones. You will need it for the trap at the end.

In [ ]:
def lorenz_x(T, dt, seed, transient=60.0):                       # GIVEN: first coordinate of the Lorenz system, standardised
    rng = np.random.default_rng(seed); s0 = np.array([1.0, 1.0, 1.0]) + 0.1 * rng.normal(size=3)
    f = lambda t, s: [10 * (s[1] - s[0]), s[0] * (28 - s[2]) - s[1], s[0] * s[1] - 8 / 3 * s[2]]
    x = solve_ivp(f, (0, transient + T), s0, t_eval=np.arange(transient, transient + T, dt), rtol=1e-9, atol=1e-11).y[0]
    return (x - x.mean()) / x.std()

def delay(x, m, tau):                                            # GIVEN: delay vectors (x_t, x_{t+tau}, ..., x_{t+(m-1)tau})
    n = len(x) - (m - 1) * tau
    return np.column_stack([x[k * tau:k * tau + n] for k in range(m)])

def two_nn(Y):                                                   # GIVEN: the TwoNN estimate of Lab 5 (ties and repeated points dropped)
    d, _ = NearestNeighbors(n_neighbors=3).fit(Y).kneighbors(Y); mu = d[:, 2] / d[:, 1]; mu = mu[np.isfinite(mu) & (mu > 1)]
    return len(mu) / np.log(mu).sum()

def autocorr_first_zero(x, max_lag=400):                         # GIVEN
    ac = np.array([np.dot(x[:-t], x[t:]) for t in range(1, max_lag)]) / np.dot(x, x)
    return int(np.argmax(ac < 0)) + 1

def mutual_information(x, lag, bins=32):
    """Mutual information (nats) between x_t and x_{t+lag}, from a 2D histogram."""
    # TODO
    raise NotImplementedError

def first_minimum(mi):
    """Lag (1, 2, ...) of the first minimum of mi, where mi[k] belongs to lag k + 1."""
    # TODO
    raise NotImplementedError

def fnn_fraction(x, m, tau, ratio=10.0, n=1500, theiler=200, seed=0):
    """Fraction of false nearest neighbours (Kennel's criterion), skipping neighbours closer than `theiler` in time."""
    # TODO
    raise NotImplementedError

def twonn_curve(x, tau, ms, n=1500, seed=0):
    """TwoNN dimension of random delay vectors, for every m in ms (mean of three draws)."""
    # TODO
    raise NotImplementedError

def twonn_consecutive_and_random(x, m, tau, n_consecutive=4000, n_random=1500, seed=0):
    """(TwoNN of the first n_consecutive delay vectors, TwoNN of n_random random delay vectors)."""
    # TODO
    raise NotImplementedError

**Part A. Which delay?** 14000 samples of $x(t)$. The first minimum of the mutual information against the first zero of the autocorrelation.

In [ ]:
# GIVEN: the series, and the two delay choices
x = lorenz_x(280, 0.02, SEED)
mi = [mutual_information(x, k) for k in range(1, 61)]
tau_star, ac_zero = first_minimum(mi), autocorr_first_zero(x)
print(f"{len(x)} samples;  first minimum of the mutual information: lag {tau_star} ({tau_star * 0.02:.2f} time units);  first zero of the autocorrelation: lag {ac_zero} ({ac_zero * 0.02:.2f} time units)")
fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(range(1, 61), mi, color=TEAL); ax.axvline(tau_star, color=AMBER, ls="--"); ax.set_xlabel("lag (samples)"); ax.set_ylabel("mutual information"); plt.tight_layout(); plt.show()

In [ ]:
# CHECK: Part A
assert 6 <= tau_star <= 10, "the first minimum of the mutual information is at about 8 samples (0.16 time units)"
assert ac_zero > 2 * tau_star, "the autocorrelation zero is far too long (and erratic: it varies a lot between runs of the same system)"
print("Exercise 4, Part A passed.")

**Part B. How many delays?** False nearest neighbours for $m=1,2,3$ with $\tau=8$.

In [ ]:
# GIVEN: false nearest neighbours
fnn = [fnn_fraction(x, m, 8) for m in (1, 2, 3)]
print("false nearest neighbours, m = 1, 2, 3:", [round(f, 3) for f in fnn])

In [ ]:
# CHECK: Part B
assert fnn[0] > 0.9, "with one coordinate almost every neighbour is false"
assert fnn[1] < 0.1, "with two coordinates most are true: the picture folds only here and there"
assert fnn[2] < 0.01, "with three coordinates it no longer folds onto itself"
print("Exercise 4, Part B passed.")

**Part C. Is it deterministic?** The dimension of the delay cloud against $m$ for four signals: Lorenz, white noise, and Lorenz with 5% observational noise.

In [ ]:
# GIVEN: saturation of the dimension
rng = np.random.default_rng(SEED + 30); ms = list(range(1, 9))
white = rng.normal(size=len(x)); xn = x + 0.05 * rng.normal(size=len(x))
curves = {"Lorenz": twonn_curve(x, 8, ms), "white noise": twonn_curve(white, 5, ms), "Lorenz + 5% noise": twonn_curve(xn, 8, ms)}
for k, v in curves.items(): print(f"{k:18s}", np.round(v, 2))
fig, ax = plt.subplots(figsize=(6.2, 3.4))
for (k, v), c in zip(curves.items(), (TEAL, MUTE, AMBER)): ax.plot(ms, v, "o-", color=c, label=k)
ax.set_xlabel("number of delays m"); ax.set_ylabel("TwoNN dimension"); ax.legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
# CHECK: Part C
L, Wn, N5 = curves["Lorenz"], curves["white noise"], curves["Lorenz + 5% noise"]
assert 0.85 < L[0] < 1.25, "one delay coordinate: a line"
assert all(1.75 < v < 2.45 for v in L[1:6]), "the Lorenz attractor saturates near 2.0 from m = 2"
assert Wn[7] > 6.0, "white noise fills whatever space it is given"
assert N5[7] > 3.3 and N5[7] > 1.4 * L[7], "observational noise breaks the saturation of a deterministic signal"
print("Exercise 4, Part C passed.")

**Part D. Noise changes the best delay.** False nearest neighbours at $m=2$ for the Lorenz series with 5% noise: $\tau=1$ against $\tau=8$.

In [ ]:
# GIVEN: delay and noise
f1, f8 = fnn_fraction(xn, 2, 1), fnn_fraction(xn, 2, 8)
print(f"5% noise, m = 2: false neighbours at tau = 1: {f1:.3f}, at tau = 8: {f8:.3f}   (without noise: {fnn_fraction(x, 2, 1):.3f} and {fnn_fraction(x, 2, 8):.3f})")

In [ ]:
# CHECK: Part D
assert f1 > f8 + 0.04, "with noise the smallest delay is no longer the best one"
print("Exercise 4, Part D passed.")

**Part E. The trap of oversampling.** The same Lorenz system sampled ten times more finely (every 0.002 time units, with $\tau=80$ samples, which is the same 0.16 time units). Take the first 4000 delay vectors, one after another, and compare with 1500 delay vectors drawn at random from the whole series.

In [ ]:
# GIVEN: the oversampled series and the two estimates
xf = lorenz_x(100, 0.002, SEED + 31)
est_consec, est_rand = twonn_consecutive_and_random(xf, 3, 80)
print(f"{len(xf)} samples;  TwoNN of 4000 consecutive vectors: {est_consec:.1f};  of 1500 random vectors: {est_rand:.2f}")

In [ ]:
# CHECK: Part E
assert est_consec > 10, "consecutive vectors: the two nearest neighbours of a point are its two temporal neighbours, at almost equal distances"
assert 1.75 < est_rand < 2.45, "random vectors from a long series: the dimension of the attractor"
print("Exercise 4 passed.")

*Discuss:* why do the two nearest neighbours of a point in an oversampled flow have almost equal distances, and what does that do to the ratio $\mu=r_2/r_1$? Why does a time window (`theiler`) alone not repair it? What did we do in the lecture instead?

## Exercise 5 — Windows as clouds (take-home)

Two windows of a time series have the same histogram of values and the same power spectrum. Do they come from the same dynamics? A window of delay vectors is a cloud, so the Wasserstein distance can compare windows.

You write:

1. `aaft(x, seed)`: the **amplitude-adjusted Fourier surrogate** of the series `x`: (a) draw a Gaussian sample of the same length, sort it, and place its values in the **rank order of `x`**, so the series has Gaussian values and the same ranks as `x`; (b) keep the Fourier amplitudes of that series (`np.fft.rfft`), replace the phases by random ones (`ph[0] = 0`, and also the last phase if the length is even), and transform back (`np.fft.irfft`); (c) place the sorted values of `x` in the rank order of the result. Ranks: `rankdata(..., method="ordinal") - 1`.
2. `window_cloud(x, k, L, m, tau, npts, seed)`: the delay vectors of window number `k` (samples `kL` to `(k+1)L`), with `npts` of them chosen at random.
3. `window_distances(x, s, m, tau, K, L, npts)`: the three lists of distances `w2_nd` between the clouds of the first `K` windows: `within_x` (all pairs of windows of `x`), `within_s` (all pairs of windows of the surrogate `s`) and `across` (every window of `x` against every window of `s`).

In [ ]:
def w2_nd(A, B):                                                 # GIVEN
    n, m = len(A), len(B)
    return float(np.sqrt(ot.emd2(np.ones(n) / n, np.ones(m) / m, ot.dist(A, B))))

def aaft(x, seed):
    """Amplitude-adjusted Fourier surrogate: the values of x, the ranks of a phase-randomised Gaussian series."""
    # TODO
    raise NotImplementedError

def window_cloud(x, k, L, m, tau, npts, seed):
    """npts random delay vectors of the window x[k L:(k + 1) L]."""
    # TODO
    raise NotImplementedError

def window_distances(x, s, m, tau, K, L, npts):
    """Lists of W2 distances: within_x, within_s and across, for the first K windows."""
    # TODO
    raise NotImplementedError

In [ ]:
# GIVEN: a long series, its surrogate, and the check that the surrogate keeps the histogram and (about) the spectrum
xl = lorenz_x(660, 0.02, SEED + 40); sl = aaft(xl, SEED + 41)
ac = lambda z, k: np.dot(z[:-k], z[k:]) / np.dot(z, z)
print(f"{len(xl)} samples;  same sorted values: {np.allclose(np.sort(xl), np.sort(sl))};  autocorrelation at lags 1, 5, 20, 50: original {[round(ac(xl, k), 2) for k in (1, 5, 20, 50)]}, surrogate {[round(ac(sl, k), 2) for k in (1, 5, 20, 50)]}")

In [ ]:
# CHECK: Part A
assert np.allclose(np.sort(xl), np.sort(sl)) and not np.allclose(xl, sl), "same values, different order"
assert max(abs(ac(xl, k) - ac(sl, k)) for k in range(1, 31)) < 0.15, "the surrogate keeps (about) the linear correlations"
print("Exercise 5, Part A passed.")

In [ ]:
# GIVEN: distances between windows: values only (m = 1) and delay vectors (m = 3); 8 windows of 4000 samples, 600 points each
res5 = {m: window_distances(xl, sl, m, 8, 8, 4000, 600) for m in (1, 3)}; ratio5 = {}
for m, (wx, ws, ac_) in res5.items():
    ratio5[m] = np.mean(ac_) / max(np.mean(wx), np.mean(ws))
    print(f"m = {m}: within the original {np.mean(wx):.3f} +- {np.std(wx):.3f} | within the surrogate {np.mean(ws):.3f} +- {np.std(ws):.3f} | original against surrogate {np.mean(ac_):.3f} +- {np.std(ac_):.3f}   ratio across / larger within = {ratio5[m]:.2f}")

In [ ]:
# CHECK: Part B
wx3, ws3, ac3 = res5[3]
assert ratio5[1] < 1.2, "the histogram of the values does not see the difference: across is not larger than within"
assert ratio5[3] > 1.3, "the delay clouds do: windows of the original and the surrogate are much further apart than windows of the same series"
assert np.mean(ac3) > max(np.mean(wx3), np.mean(ws3))
print("Exercise 5 passed.")

*Discuss:* the surrogate has the same histogram and (about) the same spectrum, so what exactly did the phase randomisation destroy, and why can the delay cloud see it? What would you change to detect a regime change in a long series (a sliding window)? The lecture slide quoted the separation in units of the within-series spread; with 8 windows that number varied between about 1 and 5 from one seed to the next, while the ratio you computed did not. Why is a spread computed from the 28 pairs of 8 windows unreliable?

## Exercise 6 — Curvature on graphs (take-home)

Ollivier's curvature of an edge $(x,y)$ is $\kappa(x,y)=1-W_1(m_x,m_y)/d(x,y)$, where $m_x$ is the uniform measure on the neighbours of $x$, and $W_1$ is computed with the shortest-path distance of the graph.

You write `ollivier_curvature(G, u, v, dist=None)`: build the matrix of shortest-path distances between the neighbours of `u` and the neighbours of `v` (from `dist`, a dictionary of dictionaries as produced by `dict(nx.all_pairs_shortest_path_length(G))`, if it is given, else with `nx.shortest_path_length`), compute $W_1$ with `ot.emd2` and uniform weights, and return $\kappa$. The function `edge_curvatures` (given) applies it to all edges.

In [ ]:
def ollivier_curvature(G, u, v, dist=None):
    """Ollivier's curvature of the edge (u, v), with m_x uniform on the neighbours of x."""
    # TODO
    raise NotImplementedError

def edge_curvatures(G):                                          # GIVEN
    dist = dict(nx.all_pairs_shortest_path_length(G))
    return {(u, v): ollivier_curvature(G, u, v, dist) for u, v in G.edges}

**Part A.** Named graphs, with values you can derive by hand (try one of them): the triangle, the complete graphs, the cycles, the grid, trees and polyhedra.

In [ ]:
# GIVEN: the named graphs
def regular_tree(d, depth):
    G = nx.Graph(); G.add_edge("a", "b"); frontier = [("a", d - 1), ("b", d - 1)]; k = 0
    for _ in range(depth):
        new = []
        for node, nch in frontier:
            for _ in range(nch): k += 1; G.add_edge(node, f"n{k}"); new.append((f"n{k}", d - 1))
        frontier = new
    return G
named = {"triangle": (nx.cycle_graph(3), 0, 1, 0.5), "K4": (nx.complete_graph(4), 0, 1, 2 / 3), "K5": (nx.complete_graph(5), 0, 1, 0.75),
         "cycle C4": (nx.cycle_graph(4), 0, 1, 0.0), "cycle C6": (nx.cycle_graph(6), 0, 1, 0.0), "grid": (nx.grid_2d_graph(9, 9), (4, 4), (4, 5), 0.0),
         "cube": (nx.hypercube_graph(3), (0, 0, 0), (0, 0, 1), 0.0), "3-regular tree": (regular_tree(3, 3), "a", "b", -2 / 3), "4-regular tree": (regular_tree(4, 3), "a", "b", -1.0),
         "5-regular tree": (regular_tree(5, 2), "a", "b", -1.2), "octahedron": (nx.octahedral_graph(), 0, 1, 0.5), "icosahedron": (nx.icosahedral_graph(), 0, 1, 0.2),
         "dodecahedron": (nx.dodecahedral_graph(), *list(nx.dodecahedral_graph().edges)[0], -1 / 3)}
kap = {name: ollivier_curvature(G, u, v) for name, (G, u, v, _) in named.items()}
for name, val in kap.items(): print(f"{name:16s} {val:+.4f}   (expected {named[name][3]:+.4f})")

In [ ]:
# CHECK: Part A
for name, val in kap.items():
    assert abs(val - named[name][3]) < 1e-9, f"{name}: expected {named[name][3]:+.4f}, got {val:+.4f}"
assert kap["4-regular tree"] < kap["grid"] < kap["K5"], "a tree is negatively curved, a grid is flat, a clique is positively curved (all with degree 4)"
print("Exercise 6, Part A passed.")

**Part B. Bridges between communities.** A stochastic block model with two blocks of 40 nodes: edges are likely inside a block (probability 0.25) and rare between blocks (0.02). Do the bridges have a different curvature?

In [ ]:
# GIVEN: the bridges of a two-community graph
G = nx.stochastic_block_model([40, 40], [[0.25, 0.02], [0.02, 0.25]], seed=SEED)
G = G.subgraph(max(nx.connected_components(G), key=len)).copy(); blk = nx.get_node_attributes(G, "block")
kc = edge_curvatures(G); intra = np.array([v for (a, b), v in kc.items() if blk[a] == blk[b]]); inter = np.array([v for (a, b), v in kc.items() if blk[a] != blk[b]])
score = -np.concatenate([intra, inter]); lab = np.concatenate([np.zeros(len(intra)), np.ones(len(inter))]); rk = rankdata(score)
auc = (rk[lab == 1].sum() - len(inter) * (len(inter) + 1) / 2) / (len(inter) * len(intra))
print(f"{len(intra)} edges inside the blocks: mean curvature {intra.mean():+.3f};  {len(inter)} edges between the blocks: mean curvature {inter.mean():+.3f};  AUC of -curvature for finding the bridges {auc:.3f}")
fig, ax = plt.subplots(figsize=(6, 3)); ax.hist(intra, bins=30, alpha=0.7, color=TEAL, label="inside"); ax.hist(inter, bins=30, alpha=0.7, color=AMBER, label="between"); ax.set_xlabel("Ollivier curvature"); ax.legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
# CHECK: Part B
assert inter.mean() < -0.3, "the bridges are strongly negative"
assert intra.mean() > -0.05, "the edges inside a community are not"
assert auc > 0.95, "the curvature alone finds the bridges"
print("Exercise 6 passed.")

*Discuss:* the dodecahedron looks like a sphere but has $\kappa=-1/3$, while the icosahedron has $+0.2$. What does the curvature of a graph really measure? What would you have to decide before computing it on the $k$-nearest-neighbour graph of a point cloud?

## Optional extension (no assert): transport on the checkpoint data

In the checkpoint project your molecules live in a space where there are no coordinates to subtract, only a **distance matrix** (for example the Tanimoto distance of the Morgan fingerprints). Optimal transport does not need coordinates: `ot.emd2(a, b, M)` takes any cost matrix `M`.

Try this on one of the datasets: take two classes, say the molecules of two different `label` values, build the sub-matrix `M = D[np.ix_(idx_class1, idx_class2)]` of the distance matrix, and compute $W_1$ between the two classes with uniform weights. Compare it with $W_1$ between two random halves of one class (the sampling floor). Do the same in the second embedding space of the checkpoint. Which classes are close, and do the two spaces agree? Write three sentences in your checkpoint report if the answer is interesting.

## What to remember

* Total variation is transport with the 0–1 cost; Wasserstein uses the distance of the data, and it is the only one of the four that keeps growing when clouds stop overlapping.
* In one dimension the optimal pairing is sorting; in general it is a linear programme, and the exact solver is the pragmatic choice up to a few thousand points. Sinkhorn needs the log domain at small $\varepsilon$.
* Between two clouds, move the points; do not mix the densities. The path of the optimal pairing is a geodesic (detour factor 1). The cloud view cannot see a rotation.
* One coordinate and its delays rebuild the dynamics: $\tau$ from the mutual information, $m$ from false neighbours, saturation of the dimension as the test. Oversampled flows need random vectors from a long series.
* A window of delay vectors is a cloud: Wasserstein sees what the histogram and the spectrum cannot.
* Curvature is a transport cost between neighbourhoods; it finds the bridges.